In [1]:
from pathlib import Path
import pandas as pd

ROOT_DIR = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

PROCESSED_DIR = ROOT_DIR / "data" / "processed"

peliculas = pd.read_csv(
    PROCESSED_DIR / "peliculas_procesadas.csv"
)

series = pd.read_csv(
    PROCESSED_DIR / "series_procesadas.csv"
)

print("Películas:", peliculas.shape)
print("Series:", series.shape)

Películas: (16000, 18)
Series: (15991, 16)


**Criterio de cálculo de la puntuación promedio**

      Para calcular la puntuación promedio se consideran únicamente los títulos que tienen al menos un voto (`vote_count > 0`). Esta decisión permite diferenciar los contenidos que cuentan con participación de usuarios de aquellos que no registran votos. Los títulos sin votos se mantienen en el catálogo y se contabilizan por separado.
      

In [6]:

def calcular_kpis(df, tipo):
    df_con_votos = df[df["vote_count"] > 0]

    return {
        "tipo_contenido": tipo,
        "total_titulos": len(df),
        "popularidad_promedio": df["popularity"].mean(),
        "puntuacion_promedio": df_con_votos["vote_average"].mean(),
        "total_votos": df["vote_count"].sum(),
        "titulos_con_votos": len(df_con_votos),
        "titulos_sin_votos": (df["vote_count"] == 0).sum()
    }

kpis = pd.DataFrame([
    calcular_kpis(peliculas, "Películas"),
    calcular_kpis(series, "Series")
])

kpis


,tipo_contenido,total_titulos,popularidad_promedio,puntuacion_promedio,total_votos,titulos_con_votos,titulos_sin_votos
0,Películas,16000,20.384728,6.308877,11498498,15106,894
1,Series,15991,64.878707,7.023750,1712228,12325,3666


In [8]:
kpis.style.format({
    "popularidad_promedio": "{:.2f}",
    "puntuacion_promedio": "{:.2f}",
    "total_titulos": "{:,.0f}",
    "total_votos": "{:,.0f}",
    "titulos_con_votos": "{:,.0f}",
    "titulos_sin_votos": "{:,.0f}"
})

,tipo_contenido,total_titulos,popularidad_promedio,puntuacion_promedio,total_votos,titulos_con_votos,titulos_sin_votos
0,Películas,"16,000",20.38,6.31,"11,498,498","15,106",894
1,Series,"15,991",64.88,7.02,"1,712,228","12,325","3,666"


In [7]:
OUTPUT_DIR = PROCESSED_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

kpis.to_csv(
    OUTPUT_DIR / "kpis_catalogo.csv",
    index=False
)

print("KPIs guardados correctamente.")

KPIs guardados correctamente.


In [4]:
for nombre, df in [
    ("Películas", peliculas),
    ("Series", series)
]:
    print(f"\n{nombre}")
    print("Puntuaciones en cero:",
          (df["vote_average"] == 0).sum())
    print("Registros sin votos:",
          (df["vote_count"] == 0).sum())
    print("Puntuaciones faltantes:",
          df["vote_average"].isna().sum())


Películas
Puntuaciones en cero: 899
Registros sin votos: 894
Puntuaciones faltantes: 0

Series
Puntuaciones en cero: 3665
Registros sin votos: 3666
Puntuaciones faltantes: 0


In [5]:
for nombre, df in [
    ("Películas", peliculas),
    ("Series", series)
]:
    promedio_original = df["vote_average"].mean()

    df_con_votos = df[df["vote_count"] > 0]
    promedio_con_votos = df_con_votos["vote_average"].mean()

    print(f"\n{nombre}")
    print(f"Promedio original: {promedio_original:.2f}")
    print(f"Promedio con votos: {promedio_con_votos:.2f}")
    print(f"Registros excluidos: {len(df) - len(df_con_votos)}")


Películas
Promedio original: 5.96
Promedio con votos: 6.31
Registros excluidos: 894

Series
Promedio original: 5.42
Promedio con votos: 7.02
Registros excluidos: 3666


In [9]:

import pandas as pd
from pathlib import Path

# Rutas de los archivos procesados
ruta_peliculas = Path("../data/processed/peliculas_procesadas.csv")
ruta_series = Path("../data/processed/series_procesadas.csv")

# Comprobar si las rutas existen desde el notebook
print("Ruta películas:", ruta_peliculas.resolve())
print("Existe:", ruta_peliculas.exists())

print("\nRuta series:", ruta_series.resolve())
print("Existe:", ruta_series.exists())

# Cargar los archivos
peliculas = pd.read_csv(ruta_peliculas)
series = pd.read_csv(ruta_series)

# Revisar dimensiones y columnas
for nombre, df in [
    ("Películas", peliculas),
    ("Series", series)
]:
    print(f"\n{'=' * 45}")
    print(f"ARCHIVO: {nombre}")
    print("Filas:", df.shape[0])
    print("Columnas:", df.shape[1])
    print("\nNombres de columnas:")
    print(df.columns.tolist())

    print("\nTipos de datos:")
    print(df.dtypes)


Ruta películas: C:\Users\nayen\EP2-VD-002V-Caso-StreamView-Analytics\data\processed\peliculas_procesadas.csv
Existe: True

Ruta series: C:\Users\nayen\EP2-VD-002V-Caso-StreamView-Analytics\data\processed\series_procesadas.csv
Existe: True

ARCHIVO: Películas
Filas: 16000
Columnas: 18

Nombres de columnas:
['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'genres', 'language', 'description', 'popularity', 'vote_count', 'vote_average', 'budget', 'revenue']

Tipos de datos:
show_id           int64
type                str
title               str
director            str
cast                str
country             str
date_added          str
release_year      int64
rating          float64
duration        float64
genres              str
language            str
description         str
popularity      float64
vote_count        int64
vote_average    float64
budget            int64
revenue           int64
dtype: object

ARCHIVO: Series

In [10]:

# Revisar valores nulos por columna
for nombre, df in [
    ("Películas", peliculas),
    ("Series", series)
]:
    print(f"\n{'=' * 50}")
    print(f"VALORES NULOS: {nombre}")

    nulos = df.isnull().sum()
    porcentaje = (nulos / len(df) * 100).round(2)

    resumen = pd.DataFrame({
        "nulos": nulos,
        "porcentaje_nulos": porcentaje
    })

    print(resumen[resumen["nulos"] > 0].sort_values(
        by="porcentaje_nulos",
        ascending=False
    ))

    print("\nTotal de celdas nulas:", df.isnull().sum().sum())



VALORES NULOS: Películas
             nulos  porcentaje_nulos
duration     16000            100.00
country        466              2.91
cast           204              1.27
director       132              0.82
description    132              0.82
genres         107              0.67

Total de celdas nulas: 17041

VALORES NULOS: Series
             nulos  porcentaje_nulos
director     10960             68.54
description   3203             20.03
country       1795             11.23
cast          1156              7.23
genres         972              6.08

Total de celdas nulas: 18086


**Tratamiento de valores faltantes**

      Se identificaron valores nulos en distintas variables descriptivas. Se conservaron los registros para evitar perder información útil para el análisis. En particular, la variable `duration` no presenta información disponible en el archivo de películas, por lo que no se considera adecuada para calcular indicadores de duración. Los valores faltantes deberán tratarse de forma explícita en las visualizaciones cuando corresponda.

In [11]:

for nombre, df in [
    ("Películas", peliculas),
    ("Series", series)
]:
    print(f"\n{'=' * 50}")
    print(f"DUPLICADOS: {nombre}")

    print("Filas duplicadas completas:", df.duplicated().sum())

    print(
        "Identificadores show_id duplicados:",
        df["show_id"].duplicated().sum()
    )

    print(
        "Títulos duplicados:",
        df["title"].duplicated().sum()
    )

    # Mostrar ejemplos de IDs repetidos, si existen
    ids_repetidos = df[
        df["show_id"].duplicated(keep=False)
    ].sort_values("show_id")

    if not ids_repetidos.empty:
        print("\nEjemplos de identificadores repetidos:")
        print(ids_repetidos[
            ["show_id", "title", "popularity", "vote_count"]
        ].head(10).to_string(index=False))
    else:
        print("No se encontraron identificadores repetidos.")



DUPLICADOS: Películas
Filas duplicadas completas: 0
Identificadores show_id duplicados: 0
Títulos duplicados: 515
No se encontraron identificadores repetidos.

DUPLICADOS: Series
Filas duplicadas completas: 0
Identificadores show_id duplicados: 0
Títulos duplicados: 323
No se encontraron identificadores repetidos.


In [12]:

import pandas as pd

for nombre, df in [
    ("Películas", peliculas),
    ("Series", series)
]:
    print(f"\n{'=' * 55}")
    print(f"VALIDACIÓN DE REGLAS DE NEGOCIO: {nombre}")

    # 1. Tipo de contenido
    print("\nTipos de contenido:")
    print(df["type"].value_counts(dropna=False))

    # 2. Años de lanzamiento
    print("\nAños de lanzamiento fuera de rango:")
    print(
        ((df["release_year"] < 1900) |
         (df["release_year"] > 2026)).sum()
    )

    # 3. Valores negativos en métricas numéricas
    columnas_numericas = [
        col for col in [
            "rating", "popularity", "vote_count",
            "vote_average", "budget", "revenue"
        ] if col in df.columns
    ]

    print("\nValores negativos:")
    for col in columnas_numericas:
        print(f"{col}: {(df[col] < 0).sum()}")

    # 4. Rangos de puntuaciones
    for col in ["rating", "vote_average"]:
        if col in df.columns:
            fuera_rango = (
                (df[col].notna()) &
                ((df[col] < 0) | (df[col] > 10))
            ).sum()
            print(f"{col} fuera del rango 0–10: {fuera_rango}")

    # 5. Fechas que no se pueden interpretar
    fechas = pd.to_datetime(df["date_added"], errors="coerce")
    fechas_invalidas = (
        df["date_added"].notna() &
        fechas.isna()
    ).sum()

    print("\nFechas no interpretables:", fechas_invalidas)
    print("Fechas faltantes:", df["date_added"].isna().sum())

    # 6. Consistencia de presupuesto e ingresos (solo películas)
    if "budget" in df.columns and "revenue" in df.columns:
        print("\nPresupuestos iguales a cero:", (df["budget"] == 0).sum())
        print("Ingresos iguales a cero:", (df["revenue"] == 0).sum())
        print("Presupuestos negativos:", (df["budget"] < 0).sum())
        print("Ingresos negativos:", (df["revenue"] < 0).sum())



VALIDACIÓN DE REGLAS DE NEGOCIO: Películas

Tipos de contenido:
type
Movie    16000
Name: count, dtype: int64

Años de lanzamiento fuera de rango:
0

Valores negativos:
rating: 0
popularity: 0
vote_count: 0
vote_average: 0
budget: 0
revenue: 0
rating fuera del rango 0–10: 0
vote_average fuera del rango 0–10: 0

Fechas no interpretables: 0
Fechas faltantes: 0

Presupuestos iguales a cero: 11153
Ingresos iguales a cero: 10355
Presupuestos negativos: 0
Ingresos negativos: 0

VALIDACIÓN DE REGLAS DE NEGOCIO: Series

Tipos de contenido:
type
TV Show    15991
Name: count, dtype: int64

Años de lanzamiento fuera de rango:
0

Valores negativos:
rating: 0
popularity: 0
vote_count: 0
vote_average: 0
rating fuera del rango 0–10: 0
vote_average fuera del rango 0–10: 0

Fechas no interpretables: 0
Fechas faltantes: 0


In [13]:

# Revisar películas con presupuesto o ingresos en cero

columnas = [
    "show_id", "title", "release_year",
    "budget", "revenue", "popularity", "vote_count"
]

print("PELÍCULAS CON PRESUPUESTO EN CERO")
display(
    peliculas.loc[peliculas["budget"] == 0, columnas].head(10)
)

print("PELÍCULAS CON INGRESOS EN CERO")
display(
    peliculas.loc[peliculas["revenue"] == 0, columnas].head(10)
)

print("PELÍCULAS CON PRESUPUESTO E INGRESOS EN CERO")
display(
    peliculas.loc[
        (peliculas["budget"] == 0) &
        (peliculas["revenue"] == 0),
        columnas
    ].head(10)
)


PELÍCULAS CON PRESUPUESTO EN CERO


,show_id,title,release_year,budget,revenue,popularity,vote_count
9,48650,Room in Rome,2010,0,844281,90.230,748
14,73861,A Serbian Film,2010,0,0,74.238,2018
17,61979,Three Steps Above Heaven,2010,0,0,66.968,3095
28,52629,El Infierno,2010,0,6703873,43.812,691
32,47931,Elite Squad: The Enemy Within,2010,0,63027681,40.660,1879
40,573531,Halo Legends,2010,0,11000000,36.147,161
44,44918,Titanic II,2010,0,0,35.090,424
49,44683,Tinker Bell and the Great Fairy Rescue,2010,0,0,33.522,945
52,48466,Scared Shrekless,2010,0,0,32.983,469
55,72641,Azureus Rising,2010,0,0,31.997,12


PELÍCULAS CON INGRESOS EN CERO


,show_id,title,release_year,budget,revenue,popularity,vote_count
14,73861,A Serbian Film,2010,0,0,74.238,2018
17,61979,Three Steps Above Heaven,2010,0,0,66.968,3095
44,44918,Titanic II,2010,0,0,35.090,424
49,44683,Tinker Bell and the Great Fairy Rescue,2010,0,0,33.522,945
52,48466,Scared Shrekless,2010,0,0,32.983,469
55,72641,Azureus Rising,2010,0,0,31.997,12
59,51620,Death Race 2,2010,7000000,0,31.427,1112
71,37707,Splice,2010,26000000,0,27.112,2273
74,534643,Tabu,2010,0,0,26.226,1
77,40205,16 Wishes,2010,0,0,26.027,1321


PELÍCULAS CON PRESUPUESTO E INGRESOS EN CERO


,show_id,title,release_year,budget,revenue,popularity,vote_count
14,73861,A Serbian Film,2010,0,0,74.238,2018
17,61979,Three Steps Above Heaven,2010,0,0,66.968,3095
44,44918,Titanic II,2010,0,0,35.090,424
49,44683,Tinker Bell and the Great Fairy Rescue,2010,0,0,33.522,945
52,48466,Scared Shrekless,2010,0,0,32.983,469
55,72641,Azureus Rising,2010,0,0,31.997,12
74,534643,Tabu,2010,0,0,26.226,1
77,40205,16 Wishes,2010,0,0,26.027,1321
82,44874,Barbie: A Fashion Fairytale,2010,0,0,25.138,760
96,50727,The Servant,2010,0,0,23.002,30


In [14]:

# Clasificación de la información financiera de películas

presupuesto_cero = peliculas["budget"].eq(0)
ingresos_cero = peliculas["revenue"].eq(0)

resumen_financiero = pd.DataFrame({
    "caso": [
        "Presupuesto e ingresos en cero",
        "Presupuesto en cero e ingresos positivos",
        "Presupuesto positivo e ingresos en cero",
        "Presupuesto e ingresos positivos"
    ],
    "cantidad": [
        (presupuesto_cero & ingresos_cero).sum(),
        (presupuesto_cero & ~ingresos_cero).sum(),
        (~presupuesto_cero & ingresos_cero).sum(),
        (~presupuesto_cero & ~ingresos_cero).sum()
    ]
})

resumen_financiero["porcentaje"] = (
    resumen_financiero["cantidad"] / len(peliculas) * 100
).round(2)

display(resumen_financiero)



,caso,cantidad,porcentaje
0,Presupuesto e ingresos en cero,9048,56.55
1,Presupuesto en cero e ingresos positivos,2105,13.16
2,Presupuesto positivo e ingresos en cero,1307,8.17
3,Presupuesto e ingresos positivos,3540,22.12


In [28]:

# PASO 5: VERIFICACIÓN FINAL DE LOS ARCHIVOS PROCESADOS

from pathlib import Path
import pandas as pd

base = Path("../data/processed")

peliculas = pd.read_csv(base / "peliculas_procesadas.csv")
series = pd.read_csv(base / "series_procesadas.csv")

print("Películas procesadas:")
print(f"Filas: {peliculas.shape[0]}")
print(f"Columnas: {peliculas.shape[1]}")

print("\nSeries procesadas:")
print(f"Filas: {series.shape[0]}")
print(f"Columnas: {series.shape[1]}")

print("\nArchivos cargados correctamente.")


Películas procesadas:
Filas: 16000
Columnas: 18

Series procesadas:
Filas: 15991
Columnas: 16

Archivos cargados correctamente.


In [29]:

# PASO 6: RESUMEN FINAL DE CALIDAD
from pathlib import Path
import pandas as pd

base = Path("../data/processed")

archivos = {
    "Películas": base / "peliculas_procesadas.csv",
    "Series": base / "series_procesadas.csv"
}

resultados = []

for nombre, ruta in archivos.items():
    df = pd.read_csv(ruta)

    resultados.append({
        "contenido": nombre,
        "filas": len(df),
        "columnas": len(df.columns),
        "duplicados_completos": int(df.duplicated().sum()),
        "show_id_duplicados": int(df["show_id"].duplicated().sum()),
        "titulos_nulos": int(df["title"].isna().sum()),
        "años_nulos": int(df["release_year"].isna().sum()),
        "años_fuera_rango": int(
            (~df["release_year"].between(1900, 2026)).sum()
        ),
        "rating_fuera_rango": int(
            (~df["rating"].dropna().between(0, 10)).sum()
        ),
        "fechas_nulas": int(df["date_added"].isna().sum())
    })

resumen_calidad = pd.DataFrame(resultados)
display(resumen_calidad)


,contenido,filas,columnas,duplicados_completos,show_id_duplicados,titulos_nulos,años_nulos,años_fuera_rango,rating_fuera_rango,fechas_nulas
0,Películas,16000,18,0,0,0,0,0,0,0
1,Series,15991,16,0,0,0,0,0,0,0
